# nano-dsv4.1f — tokenize selected 8K mid-training + SFT data

This notebook prepares stages **2 and 3** of the canonical pipeline. It does not dump entire upstream datasets into training. Selection happens **before packing**:

1. the repo catalog chooses license/quality-clean sources;
2. each adapter rejects rows that fail its source-specific verification/structure checks;
3. accepted rows are streamed only until that source's weighted token quota is reached;
4. only then are the selected traces Q-aware packed into 8192-token rows.

The resulting shards are a **materialized candidate pool**, not a command to consume every token during SFT. The final SFT step budget should be chosen after inspecting `sft_supervised_tokens` and source coverage in the manifests.


In [ ]:
from pathlib import Path
import importlib.util, json, os, shutil, subprocess, sys

TOKENIZER_DATASET = 'xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb'
REPO_URL = 'https://github.com/xiayicheng3-code/nano-dsv4.1f.git'
REPO_REF = 'main'  # printed below as an exact commit for reproducibility
HF_SECRET_NAME = 'HF_TOKEN'  # required for the official gated xLAM dataset

WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'nano-dsv4.1f'
MIDTRAIN_OUT = WORK / 'nano-dsv41f-midtrain-8k'
TRACE_OUT = WORK / 'nano-dsv41f-traces-8k'

BUILD_MIDTRAIN = True
BUILD_TRACES = True
SEQ_LEN = 8192
MIDTRAIN_STEPS = 10_000
QUERY_BUDGET = 128
Q_THRESHOLD = 640
Q_BANDS = '640,768,1024,1536,2048,3072,4096,6144,8192'
REASONING_TARGET_TOKENS = 4_000_000
AGENT_TARGET_TOKENS = 16_000_000  # soft per-source quotas; finite verified sources may exhaust
SEED = 1701

os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ['RAYON_NUM_THREADS'] = str(max(1, os.cpu_count() or 1))
print({'cpu_threads': os.environ['RAYON_NUM_THREADS'], 'seq_len': SEQ_LEN, 'stage': 'midtrain+sft'})


## Authentication + frozen tokenizer

The official Salesforce xLAM repository is gated on Hugging Face. Accept its access conditions once on Hugging Face, then add a **private Kaggle secret** named `HF_TOKEN`. Do not paste the token into this notebook.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kagglehub', 'huggingface_hub'], check=True)

if BUILD_TRACES:
    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        try:
            from kaggle_secrets import UserSecretsClient
            hf_token = UserSecretsClient().get_secret(HF_SECRET_NAME)
        except Exception as exc:
            raise RuntimeError(
                f'Missing Kaggle secret {HF_SECRET_NAME!r}. The official xLAM dataset requires authenticated HF access.'
            ) from exc
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('Hugging Face token loaded from environment/Kaggle secret (value not displayed).')

import kagglehub
tokenizer_root = Path(kagglehub.dataset_download(TOKENIZER_DATASET))
candidates = sorted(tokenizer_root.rglob('tokenizer.json'))
if not candidates:
    raise FileNotFoundError(f'No tokenizer.json found in {tokenizer_root}')
TOKENIZER_PATH = candidates[0]
print('tokenizer:', TOKENIZER_PATH)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[data]'], check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src') + os.pathsep + env.get('PYTHONPATH', '')
print('repo commit:', commit)


## Inspect the selection plan before downloading data

These are **materialization weights**, not the final SFT reasoning:agent sampling ratio. The builder stops each source once its accepted-token quota is reached. Agent quotas are soft so a finite verified source can exhaust naturally instead of being replaced with weaker data.


In [ ]:
spec = importlib.util.spec_from_file_location('trace_builder_preview', REPO_DIR / 'scripts/prepare_trace_corpus.py')
trace_builder = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = trace_builder
spec.loader.exec_module(trace_builder)

assert all(s.dataset != 'nvidia/Nemotron-SFT-Agentic-v2' for s in trace_builder.AGENT_SOURCES)
for pool_name, sources, target in (
    ('reasoning', trace_builder.REASONING_SOURCES, REASONING_TARGET_TOKENS),
    ('agent', trace_builder.AGENT_SOURCES, AGENT_TARGET_TOKENS),
):
    print(f'\n{pool_name.upper()} materialization plan — {target:,} tokens')
    for source in sources:
        print(f'  {source.key:34s} {source.weight:5.1%}  ~{round(target * source.weight):,} tokens  {source.dataset}')


## Build the single mid-training document corpus

`MIDTRAIN_STEPS` is a budget for **mid-training only**. It is not a fraction of the 3B-token pretraining run. Set `BUILD_MIDTRAIN=False` when rerunning only the SFT trace preparation.


In [ ]:
if BUILD_MIDTRAIN:
    if MIDTRAIN_OUT.exists():
        shutil.rmtree(MIDTRAIN_OUT)
    subprocess.run([
        sys.executable, str(REPO_DIR / 'scripts/prepare_midtrain_corpus.py'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--output-dir', str(MIDTRAIN_OUT),
        '--total-steps', str(MIDTRAIN_STEPS),
        '--seq-len', str(SEQ_LEN),
        '--query-budget', str(QUERY_BUDGET),
        '--q-threshold', str(Q_THRESHOLD),
        '--q-band-edges', Q_BANDS,
        '--tokenize-batch-size', '256',
        '--tokenize-batch-chars', '4000000',
        '--shard-rows', '128',
        '--seed', str(SEED),
    ], check=True, cwd=REPO_DIR, env=env)


## Build selected reasoning + agent trace shards

Do **not** concatenate entire upstream datasets first. `prepare_stage_traces.py` streams each source, applies its verifier/adapter, deduplicates rendered prompts, rejects over-length/no-supervision rows, stops at the source quota, and only then packs the accepted traces.


In [ ]:
if BUILD_TRACES:
    if TRACE_OUT.exists():
        shutil.rmtree(TRACE_OUT)
    subprocess.run([
        sys.executable, str(REPO_DIR / 'scripts/prepare_stage_traces.py'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--output-dir', str(TRACE_OUT),
        '--pool', 'all',
        '--seq-len', str(SEQ_LEN),
        '--reasoning-target-tokens', str(REASONING_TARGET_TOKENS),
        '--agent-target-tokens', str(AGENT_TARGET_TOKENS),
        '--query-budget', str(QUERY_BUDGET),
        '--q-threshold', str(Q_THRESHOLD),
        '--q-band-edges', Q_BANDS,
        '--tokenize-batch-size', '64',
        '--shuffle-buffer', '4000',
        '--shard-rows', '128',
        '--seed', str(SEED),
    ], check=True, cwd=REPO_DIR, env=env)


## Audit what actually survived selection

A source exhausting below its soft agent quota is **not** automatically back-filled from another source. Inspect coverage first. This avoids silently changing the capability mix because one verified dataset happens to be small.


In [ ]:
def source_audit(pool: str):
    manifest = json.loads((TRACE_OUT / pool / 'manifest.json').read_text())
    rows = []
    for key, info in manifest['sources'].items():
        collection = info['collection']
        requested = int(collection['target_tokens'])
        accepted = int(collection['accepted_tokens_before_effort_relabel'])
        rows.append({
            'source': key,
            'requested_tokens': requested,
            'accepted_pre_relabel': accepted,
            'final_tokens': int(info['final_tokens']),
            'final_records': int(info['final_records']),
            'coverage': accepted / requested if requested else 1.0,
            'exhausted': bool(collection.get('exhausted_before_target', False)),
            'adapter_rejected': int(collection.get('rows_adapter_or_quality_rejected', 0)),
            'too_long': int(collection.get('dropped_too_long', 0)),
            'duplicates': int(collection.get('dropped_duplicate', 0)),
        })
    return manifest, rows

if BUILD_TRACES:
    import pandas as pd
    trace_summary = json.loads((TRACE_OUT / 'trace_manifest.json').read_text())
    print('TRACE STAGE VIEWS')
    print(json.dumps(trace_summary, indent=2))
    for pool in ('reasoning', 'agent'):
        manifest, rows = source_audit(pool)
        print(f'\n{pool.upper()}')
        display(pd.DataFrame(rows))
        print({
            'records': manifest['trace_records'],
            'trace_tokens': manifest['actual_trace_tokens'],
            'sft_supervised_tokens': manifest['sft_supervised_tokens'],
            'sft_supervised_fraction': round(manifest['sft_supervised_fraction'], 4),
            'packed_rows': manifest['packing']['rows'],
            'q_budget_utilization': round(manifest['packing']['query']['mean_budget_utilization'], 4),
        })
        empty = [r['source'] for r in rows if r['final_records'] == 0]
        if empty:
            raise RuntimeError(f'{pool}: zero surviving records from {empty}')
        low = [r['source'] for r in rows if r['coverage'] < 0.25]
        if low:
            print('WARNING: <25% of nominal quota survived for:', low, '— inspect before deciding final SFT sampling.')


## What should actually go into training?

**Pack all rows that survive the pre-pack selection above; do not pack every upstream row.** The packed files retain `source_ids` per token and `sft_loss_mask` for assistant-only supervision.

For the first SFT run, keep the stage-level sampler separate from corpus construction: start from the repo's **1/3 reasoning + 2/3 agent** pool mix. Do not infer the training ratio from the 4M/16M materialization caps. If the final SFT budget is smaller than the materialized pool, shuffle and sample from these already-selected shards rather than building a second filtered dataset.

Before fixing the final number of SFT steps, use the manifest above to check: (1) supervised tokens rather than raw trace tokens, (2) actual source coverage, (3) whether any source exhausted badly, and (4) whether one source dominates after 8K packing.


In [ ]:
if BUILD_MIDTRAIN:
    curriculum = json.loads((MIDTRAIN_OUT / 'curriculum_manifest.json').read_text())
    m = json.loads((MIDTRAIN_OUT / 'midtrain' / 'manifest.json').read_text())
    print('\nMIDTRAIN SUMMARY')
    print(' rows:', m['packed']['rows'])
    print(' utilization:', round(m['packed']['real_token_utilization'], 4))
    print(' sources:', {k: round(v, 4) for k, v in m['phase']['source_weights_actual_real_tokens'].items()})
    print(' Q budget utilization:', round(m['query_packing']['mean_budget_utilization'], 4))

print('\nOUTPUT DIRECTORIES')
if BUILD_MIDTRAIN:
    print(MIDTRAIN_OUT)
if BUILD_TRACES:
    print(TRACE_OUT)
print('Save the completed directories as Kaggle Dataset outputs after reviewing the manifests above.')
